# Día 6 — Confianza por campo (PRISMA-like) para outputs estructurados

## Por qué existe este notebook
En sistemas tipo PRISMA (OCR + extracción + IA), el resultado final no es “una predicción”.
El resultado es un **objeto estructurado** con múltiples campos (por ejemplo: número de factura,
fecha, total, moneda, proveedor, impuestos, etc.).

El problema operativo es que el riesgo **no está en el documento completo**:
está en **cada campo**.

Un documento puede verse “bien” en general, pero fallar en un campo crítico como:
- `total_amount` (impacto financiero directo),
- `invoice_number` (trazabilidad),
- `tax_id` (cumplimiento),
- `date` (contabilidad y auditoría).

Por eso, una confianza global “promedio” es insuficiente.
Necesitamos una **confidence layer por campo**.

---

## Objetivo del notebook
Construir un prototipo funcional que, para cada campo extraído, genere:

1) Un **paquete de scores** (señales) por campo  
2) Un **score compuesto** de confianza por campo  
3) Un `flag_review` por campo (revisión selectiva)  
4) Un `flag_review_global` para el documento completo (decisión final)

El output final debe parecerse a esto:

```json
{
  "document_id": "doc_001",
  "fields": [
    {
      "field": "total_amount",
      "value": 1234.56,
      "signals": {"p_max": 0.91, "entropy": 0.22, "variability": 0.04},
      "confidence_score": 0.74,
      "flag_review": false
    },
    ...
  ],
  "flag_review_global": false
}

## Qué vamos a simular (porque todavía no es el OCR real)

Para poder diseñar y validar la lógica sin depender del OCR/LLM,
vamos a **simular** un output PRISMA-like:

* `field`: nombre del campo
* `value`: valor propuesto
* `scores/signals`: señales de confianza e incertidumbre

Esto nos permite iterar rápido y probar reglas de negocio antes de integrarlo
con el extractor real.

---

## Señales que vamos a usar por campo (y para qué sirven)

Cada campo tendrá un conjunto de señales:

### 1) `p_max` (confidence)

Qué tan fuerte “gana” el valor propuesto (probabilidad top-1).
Sirve para estimar **confianza directa**, pero por sí sola puede ser engañosa.

### 2) `entropy` (ambigüedad)

Mide si el modelo está repartiendo probabilidad entre varias alternativas.
Alta entropía suele indicar “hay varias respuestas plausibles”.
Sirve para detectar **confusión**, especialmente en multiclase o selección de candidatos.

### 3) `variability` (estabilidad)

Mide si el score cambia mucho al repetir el modelo (ensemble/seed/submuestras).
Alta variabilidad indica que la predicción es **frágil**.
Sirve para detectar **riesgo estructural**, aunque `p_max` sea alto.

---

## Calibración por campo (la parte más importante)

Aunque uses “el mismo sistema”, cada campo tiene comportamientos distintos:

* algunos son fáciles (moneda, fecha en formato estándar),
* otros son difíciles (proveedor, direcciones, totales con OCR ruidoso).

Eso significa que la relación entre `p_max` y “acierto real” puede variar por campo.
Por lo tanto:

* calibramos y evaluamos por campo,
* definimos umbrales y políticas por campo (o por criticidad).

En negocio: **no se gestiona riesgo igual para un `supplier_name` que para un `total_amount`.**

---

## Score compuesto por campo (confidence_score)

Vamos a construir un score compuesto que combine:

* confianza (`p_max`, idealmente calibrada),
* penalización por ambigüedad (entropía),
* penalización por inestabilidad (variabilidad).

El objetivo no es encontrar “la fórmula perfecta”,
sino producir un score que sea:

* **explicable**
* **estable**
* **accionable**
* **auditable**

---

## Flags de revisión (campo y global)

### `flag_review` por campo

Se activa cuando el campo no cumple el estándar requerido, por ejemplo:

* p_max bajo,
* entropía alta,
* variabilidad alta,
* o reglas específicas (campos críticos más estrictos).

### `flag_review_global`

Se calcula agregando los flags de campo, por ejemplo:

* “si falla un campo crítico, revisar el documento”
* “si fallan ≥ N campos, revisar”
* “si el score promedio ponderado cae debajo de X”

---

## Resultado esperado (output)

Al finalizar, tendremos un **prototipo de confidence layer** que:

* produce un JSON por documento con señales y confianza por campo,
* marca revisión selectiva (mejor costo/beneficio),
* crea trazabilidad (por qué se revisa),
* es directamente integrable en PRISMA/agents (como gating de decisiones).

**Este notebook convierte la extracción en un proceso gobernado por riesgo,
no por intuición.**

```
```


---

## Simulación de un output PRISMA-like (estructura + señales)

### Qué vamos a hacer
Vamos a crear un **dataset sintético** que imite el output de un sistema tipo PRISMA:

- Un conjunto de **documentos**
- Cada documento contiene varios **campos** (`field`)
- Cada campo tiene:
  - un `value` (valor propuesto)
  - señales (`signals`) para gobernar confianza:
    - `p_max` (confidence)
    - `entropy_norm` (ambigüedad)
    - `std_pmax` (variabilidad)

### Por qué lo hacemos así
Porque antes de integrar OCR/LLM, necesitamos una base controlada para:
- probar la lógica de **confianza por campo**
- simular campos fáciles vs difíciles
- observar cómo cambian las decisiones (`flag_review`)
- iterar rápido los umbrales y reglas

### Qué vamos a simular exactamente
1) **Campos con comportamientos distintos**:
   - `invoice_number`: normalmente claro, pero sensible a OCR.
   - `date`: suele ser estable.
   - `total_amount`: crítico y propenso a errores.
   - `currency`: fácil casi siempre.
   - `supplier_name`: difícil (muchas variantes).

2) **Distribuciones realistas de señales**:
   - Campos “fáciles” tienden a:
     - p_max alto
     - entropía baja
     - variabilidad baja
   - Campos “difíciles” tienden a:
     - p_max más bajo
     - entropía alta
     - variabilidad más alta

### Output de esta celda
Un DataFrame con filas por campo y columnas como:
- `document_id`
- `field`
- `value`
- `p_max`
- `entropy_norm`
- `std_pmax`

Ese DataFrame será la base para:
- calibración por campo (día 6)
- score compuesto por campo
- flags por campo + global

---
---
---

## Celda 2.1 — Setup: imports, configuración y catálogo de campos

Este bloque prepara el entorno para la simulación PRISMA-like:
- Importa librerías (`numpy`, `pandas`)
- Define el generador aleatorio (reproducible)
- Declara la lista de campos que tendrá cada documento
- Define una heurística de “dificultad” por campo (0 fácil → 1 difícil)

Esa dificultad controlará cómo se simulan las señales:
- campos más difíciles → menor `p_max`, mayor `entropy_norm`, mayor `std_pmax`

In [2]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)

FIELDS = ["invoice_number", "date", "total_amount", "currency", "supplier_name"]

# "Dificultad" por campo: 0 fácil → 1 difícil
FIELD_DIFFICULTY = {
    "invoice_number": 0.45,
    "date": 0.20,
    "total_amount": 0.55,
    "currency": 0.10,
    "supplier_name": 0.70,
}

## Celda 2.2 — Simular valores por campo (value)

Este bloque crea valores “de juguete” para cada campo, solo para que el output
se parezca a un JSON real de PRISMA.

Importante:
- Estos valores NO son el foco del notebook.
- El foco es la capa de confianza (`p_max`, `entropy_norm`, `std_pmax`).

In [3]:
def simulate_value(field, doc_id):
    if field == "invoice_number":
        return f"INV-{doc_id:03d}-{rng.integers(1000,9999)}"
    if field == "date":
        day = int(rng.integers(1, 29))
        month = int(rng.integers(1, 13))
        year = 2025
        return f"{year}-{month:02d}-{day:02d}"
    if field == "total_amount":
        return float(np.round(rng.uniform(10, 5000), 2))
    if field == "currency":
        return rng.choice(["EUR", "USD", "GBP"])
    if field == "supplier_name":
        return rng.choice(["ACME Corp", "Globex", "Initech", "Umbrella LLC", "Soylent Co"])
    return None

## Celda 2.3 — Simular señales por campo (p_max, entropy_norm, std_pmax)

Este bloque genera las señales que usarán la confidence layer:

- `p_max`: confianza top-1 (más alto = mejor)
- `entropy_norm`: ambigüedad (más alto = más confusión)
- `std_pmax`: variabilidad (más alto = menos estabilidad)

La dificultad del campo controla las distribuciones:
- campo difícil ⇒ `p_max` baja, `entropy_norm` sube, `std_pmax` sube

Esto imita el comportamiento real: ciertos campos son intrínsecamente
más ruidosos o ambiguos.

In [4]:
def simulate_signals(field):
    d = FIELD_DIFFICULTY[field]

    # p_max: más bajo si el campo es más difícil
    base_p = 0.92 - 0.35 * d
    p_max = float(np.clip(rng.normal(loc=base_p, scale=0.08 + 0.10*d), 0.05, 0.99))

    # entropía norm: más alta si el campo es más difícil
    base_h = 0.15 + 0.75 * d
    entropy_norm = float(np.clip(rng.normal(loc=base_h, scale=0.10 + 0.08*d), 0.0, 1.0))

    # variabilidad: más alta si el campo es más difícil
    base_std = 0.01 + 0.12 * d
    std_pmax = float(np.clip(rng.normal(loc=base_std, scale=0.01 + 0.02*d), 0.0, 0.30))

    return p_max, entropy_norm, std_pmax

## Celda 2.4 — Generar dataset PRISMA-like (documentos × campos)

Este bloque crea un DataFrame con una fila por campo, por documento:
- `document_id`
- `field`
- `value`
- señales: `p_max`, `entropy_norm`, `std_pmax`

Este DataFrame será la base para:
- calibración por campo
- score compuesto por campo
- `flag_review` por campo y global

In [ ]:
N_DOCS = 20

rows = []
for doc_id in range(1, N_DOCS + 1):
    for field in FIELDS:
        value = simulate_value(field, doc_id)
        p_max, entropy_norm, std_pmax = simulate_signals(field)

        rows.append({
            "document_id": f"doc_{doc_id:03d}",
            "field": field,
            "value": value,
            "p_max": round(p_max, 3),
            "entropy_norm": round(entropy_norm, 3),
            "std_pmax": round(std_pmax, 3),
        })

df_fields = pd.DataFrame(rows)

df_fields.head(10)

,document_id,field,value,p_max,entropy_norm,std_pmax
0,doc_001,invoice_number,INV-001-3021,0.742,0.773,0.036
1,doc_001,date,2025-10-26,0.990,0.352,0.042
2,doc_001,total_amount,4415.3,0.772,0.674,0.059
3,doc_001,currency,EUR,0.964,0.077,0.026
4,doc_001,supplier_name,Globex,0.583,0.671,0.079
...,...,...,...,...,...,...
95,doc_020,invoice_number,INV-020-8738,0.653,0.579,0.080
96,doc_020,date,2025-12-07,0.762,0.274,0.044
97,doc_020,total_amount,1352.96,0.577,0.622,0.088
98,doc_020,currency,EUR,0.905,0.197,0.040


## Interpretación del resultado — columna por columna

### `document_id`
Identificador del documento procesado.  
Agrupa todos los campos que pertenecen al mismo documento (ej. `doc_001`).

---

### `field`
Nombre del campo extraído del documento.  
Cada fila representa **un campo independiente** (no el documento completo).

Ejemplos:
- `invoice_number`
- `date`
- `total_amount`
- `currency`
- `supplier_name`

---

### `value`
Valor propuesto por el sistema para ese campo.  
Es el **resultado funcional** que luego se usa en sistemas contables, fiscales o de negocio.

Ejemplo:
- `2461.37` para `total_amount`
- `USD` para `currency`

---

### `p_max`
**Confidence** del campo.  
Es la probabilidad más alta asignada al valor elegido.

Lectura:
- Alto (`≈0.9`) → el modelo “cree” firmemente en ese valor.
- Medio (`≈0.6–0.7`) → confianza limitada.
- Bajo → señal débil, no automatizable.

---

### `entropy_norm`
**Entropía normalizada** del campo.  
Mide **ambigüedad** entre alternativas posibles.

Lectura:
- Baja (`≈0.2–0.3`) → decisión clara.
- Media (`≈0.4–0.6`) → competencia moderada.
- Alta (`≥0.65`) → confusión real entre opciones.

---

### `std_pmax`
**Variabilidad** de la confianza.  
Mide qué tan estable es el score cuando se repite la predicción.

Lectura:
- Baja (`<0.03`) → predicción estable.
- Media (`0.03–0.07`) → cierta fragilidad.
- Alta (`>0.08`) → el modelo cambia mucho de opinión.

---
---
---

## Análisis y calibración **por campo**

### Qué vamos a hacer en esta celda
Vamos a **agrupar los resultados por campo** para entender su comportamiento estadístico
antes de tomar decisiones.

El objetivo no es decidir aún, sino **diagnosticar**.

Para cada `field` vamos a responder:
- ¿Qué tan confiable suele ser (`p_max`)?
- ¿Qué tan ambiguo suele ser (`entropy_norm`)?
- ¿Qué tan estable es el modelo (`std_pmax`)?

Esto nos permite:
- confirmar que los campos “difíciles” realmente se comportan como tales,
- justificar **umbrales distintos por campo**,
- preparar la calibración y las reglas del score compuesto.

### Por qué esto es crítico
Un error común es usar **los mismos thresholds para todos los campos**.
Eso es incorrecto porque:
- los campos no tienen el mismo nivel de ruido,
- no tienen el mismo impacto de negocio,
- no tienen la misma dificultad semántica.

Esta celda convierte intuiciones en **evidencia cuantitativa**.


In [10]:
# CELDA 3 — Estadísticas por campo (diagnóstico)

field_stats = (
    df_fields
    .groupby("field")
    .agg(
        p_max_mean=("p_max", "mean"),
        p_max_std=("p_max", "std"),
        entropy_mean=("entropy_norm", "mean"),
        entropy_std=("entropy_norm", "std"),
        variability_mean=("std_pmax", "mean"),
        variability_std=("std_pmax", "std"),
        n_samples=("p_max", "count"),
    )
    .reset_index()
)

# Redondeamos para lectura clara
field_stats = field_stats.round(3)

field_stats

,field,p_max_mean,p_max_std,entropy_mean,entropy_std,variability_mean,variability_std,n_samples
0,currency,0.888,0.067,0.203,0.102,0.019,0.012,20
1,date,0.841,0.109,0.313,0.122,0.032,0.012,20
2,invoice_number,0.758,0.132,0.463,0.160,0.060,0.019,20
3,supplier_name,0.688,0.130,0.629,0.158,0.093,0.024,20
4,total_amount,0.712,0.128,0.552,0.145,0.080,0.013,20


## Interpretación de los resultados — análisis por campo

Estos resultados resumen el **comportamiento promedio** de cada campo a lo largo
de los documentos simulados. No describen un caso puntual, sino la **naturaleza
estructural** de cada campo.

---

### `currency`
- **p_max_mean = 0.888** (alto)
- **entropy_mean = 0.203** (baja)
- **variability_mean = 0.019** (muy baja)

**Lectura:**  
Campo **altamente confiable**. El modelo suele estar seguro, no se confunde y es
estable.  
Es un claro candidato a **automatización directa** con umbrales laxos.

---

### `date`
- **p_max_mean = 0.841** (alto)
- **entropy_mean = 0.313** (baja–media)
- **variability_mean = 0.032** (baja)

**Lectura:**  
Campo generalmente confiable, con algo más de ruido que `currency`.  
Aún así, es **automatizable** en la mayoría de los casos, con controles mínimos.

---

### `invoice_number`
- **p_max_mean = 0.758** (medio)
- **entropy_mean = 0.463** (media)
- **variability_mean = 0.060** (media)

**Lectura:**  
Campo de **dificultad intermedia**.  
El modelo acierta con frecuencia, pero muestra:
- confusión ocasional,
- sensibilidad a OCR/formato.

Requiere **umbrales más estrictos** y revisión selectiva.

---

### `supplier_name`
- **p_max_mean = 0.688** (medio–bajo)
- **entropy_mean = 0.629** (alta)
- **variability_mean = 0.093** (alta)

**Lectura:**  
Campo **intrínsecamente difícil**:
- muchas variantes válidas,
- ambigüedad semántica,
- alta inestabilidad.

Es el principal candidato a **zona gris permanente** o revisión frecuente.

---

### `total_amount`
- **p_max_mean = 0.712** (medio)
- **entropy_mean = 0.552** (media–alta)
- **variability_mean = 0.080** (media–alta)

**Lectura:**  
Campo **crítico y sensible**.  
Aunque la confianza promedio no es baja, la combinación de ambigüedad y
variabilidad lo vuelve **riesgoso de automatizar sin control**.

Debe tener reglas **más conservadoras** por impacto financiero.

---

## Conclusión general

1. **Los campos no se comportan igual**  
   Cada campo tiene una “firma de incertidumbre” propia.

2. **La confianza global sería engañosa**  
   Un documento puede ser confiable en `currency` y riesgoso en `total_amount`.

3. **Se justifica la calibración por campo**  
   Estos resultados respaldan:
   - umbrales distintos,
   - pesos distintos,
   - políticas de revisión específicas.

4. **Base sólida para la siguiente celda**  
   Con este diagnóstico, ya es seguro construir:
   - el **score compuesto por campo**,
   - los `flag_review` con criterio técnico y de negocio.

Este análisis convierte intuición en **gobernanza cuantificable**.

---
---
---

## Score compuesto por campo (confidence_score)

### Qué problema resolvemos aquí
Hasta ahora tenemos **señales separadas** por campo:
- `p_max` → qué tan fuerte es la predicción
- `entropy_norm` → qué tan confundido está el modelo
- `std_pmax` → qué tan estable es la predicción

El problema es que **no se decide con señales aisladas**.
En producción necesitamos **una señal única**, clara y accionable por campo.

Eso es el **score compuesto por campo**.

---

## Principio de diseño (muy importante)

No buscamos:
- optimizar una métrica,
- ni una fórmula matemática “óptima”.

Buscamos un score que sea:
- **explicable** (entendible por negocio),
- **consistente** (comportamiento predecible),
- **operable** (sirva para activar reglas).

---

## Idea conceptual del score

La lógica es deliberadamente simple:

> **Confianza por campo =**
> - confianza directa  
> - penalización por ambigüedad  
> - penalización por inestabilidad  

Es decir:
- una probabilidad alta **no basta**
- si hay confusión o fragilidad, el score baja

---

## Normalización y pesos
Para poder combinar señales:
- todas deben estar en el rango **[0, 1]**
- los pesos reflejan **criterio de negocio**, no estadística pura

Ejemplo de pesos iniciales:
- confianza (`p_max`) → peso mayor
- entropía y variabilidad → penalizaciones

Estos pesos **se ajustan**, no se “aprenden”.


---

### Qué hace este código (resumen)

Este código construye un **score único de confianza por campo** combinando tres señales:

- **Confianza directa (`p_max`)**: qué tan fuerte es la predicción.
- **Entropía (`entropy_norm`)**: cuánta ambigüedad existe entre alternativas.
- **Variabilidad (`std_pmax`)**: qué tan estable es la predicción.

La lógica es deliberadamente simple:
- la confianza **suma**,
- la ambigüedad y la inestabilidad **restan**.

El resultado es un `confidence_score` entre **0 y 1** que:
- penaliza predicciones engañosamente seguras,
- prioriza estabilidad y claridad,
- permite tomar decisiones operativas por campo
  (automatizar, revisar o escalar).

Este score convierte señales estadísticas en una **señal accionable de negocio**.

In [11]:
# CELDA 4 — Cálculo del score compuesto por campo

# Pesos iniciales (criterio operativo, no mágico)
W_CONF = 0.60      # peso de la confianza directa
W_ENTROPY = 0.25   # penalización por ambigüedad
W_VARIABILITY = 0.15  # penalización por inestabilidad

def compute_confidence_score(p_max, entropy_norm, std_pmax):
    """
    Score compuesto por campo.
    Retorna un valor en [0, 1].
    """
    score = (
        W_CONF * p_max
        - W_ENTROPY * entropy_norm
        - W_VARIABILITY * std_pmax
    )
    # Aseguramos rango válido
    return float(np.clip(score, 0.0, 1.0))

# Aplicamos el score a cada fila
df_fields["confidence_score"] = df_fields.apply(
    lambda r: compute_confidence_score(
        r["p_max"], r["entropy_norm"], r["std_pmax"]
    ),
    axis=1
)

# Vista rápida
df_fields.head(10)

,document_id,field,value,p_max,entropy_norm,std_pmax,confidence_score
0,doc_001,invoice_number,INV-001-3021,0.742,0.773,0.036,0.24655
1,doc_001,date,2025-10-26,0.990,0.352,0.042,0.49970
2,doc_001,total_amount,4415.3,0.772,0.674,0.059,0.28585
3,doc_001,currency,EUR,0.964,0.077,0.026,0.55525
4,doc_001,supplier_name,Globex,0.583,0.671,0.079,0.17020
5,doc_002,invoice_number,INV-002-3460,0.629,0.439,0.043,0.26120
6,doc_002,date,2025-10-07,0.735,0.383,0.017,0.34270
7,doc_002,total_amount,4104.58,0.643,0.753,0.083,0.18510
8,doc_002,currency,EUR,0.930,0.000,0.021,0.55485
9,doc_002,supplier_name,Initech,0.975,0.696,0.096,0.39660


### Interpretación breve de los resultados (score compuesto)

Estos resultados muestran cómo el **score compuesto por campo** ajusta la confianza
real, más allá de la probabilidad (`p_max`).

- **currency**  
  Presenta los **scores más altos** (`≈0.55`).  
  Combina p_max alto con entropía y variabilidad bajas → **campo claramente automatizable**.

- **date**  
  Scores medios (`≈0.34–0.50`).  
  Aunque p_max puede ser alto, cierta ambigüedad reduce la confianza final → **automatizable con control ligero**.

- **invoice_number**  
  Scores medios-bajos (`≈0.25–0.26`).  
  La entropía elevada penaliza la confianza → **requiere revisión selectiva**.

- **total_amount**  
  Scores bajos (`≈0.18–0.29`).  
  A pesar de p_max aceptable, la alta entropía y variabilidad lo vuelven **riesgoso** → **no automatizar sin control**.

- **supplier_name**  
  Los **scores más bajos** (`≈0.17–0.40`).  
  Campo semánticamente difícil y poco estable → **zona gris estructural**.

**Conclusión:**  
El `confidence_score` cumple su objetivo: **castiga la falsa seguridad** y prioriza
campos estables y claros. Esto habilita decisiones por campo en lugar de confiar
ciegamente en `p_max`.

---
---
---

### Qué vamos a hacer en esta celda

Ahora que ya tenemos un **confidence_score por campo**, vamos a convertir ese
score en una **decisión operativa clara**.

El objetivo es simple:
- decidir **campo por campo** si el valor puede usarse automáticamente,
- o si debe ir a **revisión**.

Aquí no buscamos sofisticación matemática.
Buscamos **reglas claras, explicables y auditables**.

---

### Principio clave

> No todos los campos tienen el mismo riesgo.

Por eso:
- algunos campos pueden tolerar menor confianza,
- otros (financieros o regulatorios) requieren reglas más estrictas.

Esta celda introduce:
- **umbrales por campo**
- un `flag_review` booleano por campo

Esto es lo que habilita:
- revisión selectiva,
- UI con highlight por campo,
- reducción de costos operativos.

In [12]:
# CELDA 5 — Flag de revisión por campo

# Umbrales por campo (ejemplo inicial, ajustables por negocio)
FIELD_THRESHOLDS = {
    "currency": 0.45,
    "date": 0.40,
    "invoice_number": 0.35,
    "total_amount": 0.50,     # más estricto por impacto financiero
    "supplier_name": 0.55,    # más estricto por ambigüedad semántica
}

def flag_review_field(field, confidence_score):
    """
    Devuelve True si el campo requiere revisión.
    """
    threshold = FIELD_THRESHOLDS.get(field, 0.5)
    return confidence_score < threshold

# Aplicamos flag por campo
df_fields["flag_review"] = df_fields.apply(
    lambda r: flag_review_field(r["field"], r["confidence_score"]),
    axis=1
)

# Vista rápida
df_fields.head(10)

,document_id,field,value,p_max,entropy_norm,std_pmax,confidence_score,flag_review
0,doc_001,invoice_number,INV-001-3021,0.742,0.773,0.036,0.24655,True
1,doc_001,date,2025-10-26,0.990,0.352,0.042,0.49970,False
2,doc_001,total_amount,4415.3,0.772,0.674,0.059,0.28585,True
3,doc_001,currency,EUR,0.964,0.077,0.026,0.55525,False
4,doc_001,supplier_name,Globex,0.583,0.671,0.079,0.17020,True
5,doc_002,invoice_number,INV-002-3460,0.629,0.439,0.043,0.26120,True
6,doc_002,date,2025-10-07,0.735,0.383,0.017,0.34270,True
7,doc_002,total_amount,4104.58,0.643,0.753,0.083,0.18510,True
8,doc_002,currency,EUR,0.930,0.000,0.021,0.55485,False
9,doc_002,supplier_name,Initech,0.975,0.696,0.096,0.39660,True


**Resumen operativo de los resultados**

* **`flag_review = True`** significa que **ese campo no cumple el umbral mínimo de confianza definido para su tipo**.
  No implica que esté mal, implica que **no es suficientemente confiable para automatizar** sin revisión.

* En ambos documentos:

  * **`currency`** y **algunas fechas** aparecen con `flag_review = False`
    → campos claros, estables y automatizables.
  * **`invoice_number`, `total_amount` y `supplier_name`** aparecen mayoritariamente con `flag_review = True`
    → alta ambigüedad, inestabilidad o criticidad.

* Casos clave:

  * **`total_amount`**: aunque `p_max` es aceptable, la **entropía y variabilidad altas** reducen el score → revisión obligatoria.
  * **`supplier_name`**: incluso con `p_max` alto (doc_002), la **ambigüedad semántica y la inestabilidad** activan revisión.
  * **`date` (doc_002)**: score por debajo del umbral específico del campo → revisión puntual.

**Conclusión**
El `flag_review` convierte señales estadísticas en **acciones claras por campo**.
La revisión es **selectiva y justificada**, no global ni arbitraria.

---
---
---

### Qué vamos a hacer en esta celda

Vamos a pasar de decisiones **por campo** a una decisión **por documento**.

El objetivo es responder una sola pregunta operativa:
> ¿Este documento puede fluir automáticamente o requiere revisión?

La idea es **no perder granularidad**:
- los campos siguen teniendo su `flag_review`,
- pero ahora definimos una **regla clara de agregación**.

---

### Principio de negocio

No todos los campos pesan igual.
Por ejemplo:
- un error en `currency` puede ser tolerable,
- un error en `total_amount` no.

Por eso, la decisión global se basa en:
- **campos críticos**, y/o
- **cantidad de campos en revisión**.

---

### Estrategias típicas (usaremos una simple y defendible)

1) **Regla por campo crítico (OR)**  
   Si falla un campo crítico → revisar documento.

2) **Regla por volumen**  
   Si fallan ≥ N campos → revisar documento.

En este prototipo aplicamos **ambas**.

In [13]:
# CELDA 6 — Flag de revisión global por documento

# Definimos campos críticos (ejemplo)
CRITICAL_FIELDS = {"total_amount", "supplier_name"}

# Umbral de cantidad de campos en revisión
MAX_REVIEW_FIELDS = 2

def flag_review_global(doc_df):
    """
    Decide si un documento requiere revisión global.
    """
    # Regla 1: falla algún campo crítico
    if any((doc_df["field"].isin(CRITICAL_FIELDS)) & (doc_df["flag_review"])):
        return True

    # Regla 2: demasiados campos en revisión
    if doc_df["flag_review"].sum() > MAX_REVIEW_FIELDS:
        return True

    return False

# Aplicamos por documento
global_flags = (
    df_fields
    .groupby("document_id")
    .apply(flag_review_global)
    .reset_index(name="flag_review_global")
)

# Unimos al dataframe original
df_fields = df_fields.merge(global_flags, on="document_id", how="left")

# Vista rápida
df_fields.head(15)

C:\Users\Jorge Luis\AppData\Local\Temp\ipykernel_2476\467448190.py:27: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(flag_review_global)


,document_id,field,value,p_max,entropy_norm,std_pmax,confidence_score,flag_review,flag_review_global
0,doc_001,invoice_number,INV-001-3021,0.742,0.773,0.036,0.24655,True,True
1,doc_001,date,2025-10-26,0.990,0.352,0.042,0.49970,False,True
2,doc_001,total_amount,4415.3,0.772,0.674,0.059,0.28585,True,True
3,doc_001,currency,EUR,0.964,0.077,0.026,0.55525,False,True
4,doc_001,supplier_name,Globex,0.583,0.671,0.079,0.17020,True,True
5,doc_002,invoice_number,INV-002-3460,0.629,0.439,0.043,0.26120,True,True
6,doc_002,date,2025-10-07,0.735,0.383,0.017,0.34270,True,True
7,doc_002,total_amount,4104.58,0.643,0.753,0.083,0.18510,True,True
8,doc_002,currency,EUR,0.930,0.000,0.021,0.55485,False,True
9,doc_002,supplier_name,Initech,0.975,0.696,0.096,0.39660,True,True


**Diagnóstico breve del `flag_review_global`**

* En los **tres documentos** (`doc_001`, `doc_002`, `doc_003`), el `flag_review_global` es **True**.
* La causa no es un fallo general del modelo, sino la presencia de **campos críticos** con `flag_review = True`, principalmente:

  * **`total_amount`** (impacto financiero),
  * **`supplier_name`** (alta ambigüedad e inestabilidad).
* Aunque algunos campos como **`currency`** e incluso **`date`** muestran buena confianza, **no compensan** el riesgo introducido por los campos críticos.
* El sistema actúa correctamente: **bloquea la automatización completa** y fuerza revisión cuando el riesgo es relevante, en lugar de promediar confianza y ocultar problemas.

**Conclusión:**
El `flag_review_global` está funcionando como un **mecanismo de control de riesgo**: permite automatizar solo cuando **ningún campo crítico compromete el documento**.